# Team02 — S3 Polling Trigger for the Final SageMaker Champion Pipeline

**Updated to align with:** `S201_1552444F_03_sagemaker_pipeline_team02_champion_model_UPDATED_FINAL_v5.ipynb`

## Purpose

This notebook provides the **AWS-only S3 change-trigger demo path** for Team02. It watches a controlled S3 folder and starts the existing SageMaker Pipeline when a valid new CSV object is detected.

### Final v5 pipeline contract

| Item | Final configuration |
|---|---|
| AWS Region | `ap-southeast-1` |
| Bucket | `nyp-26s1-iti113` |
| Team | `team02` |
| SageMaker Pipeline | `team02-diabetes-risk` |
| Final endpoint | `team02-diabetes-risk` |
| Final model package group | `team02-diabetes-risk` |
| Champion | **XGBoost** |
| Champion final experiment run | `057bb58d81564c0db9a628b224eab8f9` |
| Champion config version | `2026-08-22-cross-model-champion-v5` |
| Data version | `brfss2015-diabetes-binary-6244bec277fe` |
| Trigger folder | `s3://nyp-26s1-iti113/iti113/team02/trigger/input/` |

## Important governance behavior

The v5 pipeline is deliberately **lineage-frozen**. A trigger file is accepted only when it reproduces the approved full modelling dataset and deterministic train/test split signatures. This prevents an arbitrary or corrupted CSV from silently retraining and registering a model.

The trigger starts the pipeline only. It **does not automatically approve a model package or deploy it to the final endpoint**. Registration remains `PendingManualApproval`, preserving the v5 human-over-the-loop release control.

### End-to-end position

`S3 upload → polling detection → data-lineage preflight → SageMaker Pipeline → preprocessing → XGBoost training → evaluation → quality gate → PendingManualApproval registration → human review/approval → deployment → inference → monitoring`


## 1. Human Control Panel

For a normal demonstration:

1. Leave `PROCESS_EXISTING_OBJECTS = False`.
2. Run the notebook through the initialization cell.
3. Set `RUN_CONTINUOUS_POLLING = True`.
4. Run the polling cell.
5. Upload a **copy of the approved cleaned Team02 dataset** into the trigger folder.

`PROCESS_EXISTING_OBJECTS=False` is intentional: files already in the folder when the notebook starts are treated as the baseline and are not retriggered accidentally.


In [1]:
# ============================================================
# TEAM02 S3 POLLING TRIGGER — HUMAN CONTROL PANEL
# ============================================================

CHECK_INTERVAL_SECONDS = 60

# False = ignore objects already present when this notebook starts.
# True  = treat all current objects as new candidates on the first scan.
PROCESS_EXISTING_OBJECTS = False

# Keep False while validating the notebook.
# Set True only when you intentionally want the cell to remain polling.
RUN_CONTINUOUS_POLLING = False

# Strong safety control: block triggering an AWS pipeline unless its
# tags/definition match the final v5 Team02 champion pipeline contract.
REQUIRE_FINAL_V5_PIPELINE = True

print("Human Control Panel")
print("-" * 72)
print("Polling interval (seconds) :", CHECK_INTERVAL_SECONDS)
print("Process existing objects  :", PROCESS_EXISTING_OBJECTS)
print("Continuous polling        :", RUN_CONTINUOUS_POLLING)
print("Require final v5 pipeline :", REQUIRE_FINAL_V5_PIPELINE)


Human Control Panel
------------------------------------------------------------------------
Polling interval (seconds) : 60
Process existing objects  : False
Continuous polling        : False
Require final v5 pipeline : True


## 2. Final v5 Configuration and Frozen Lineage

These constants mirror Notebook 03 v5. The trigger sends the same pipeline parameters used by the v5 notebook when starting the final champion pipeline.


In [2]:
import io
import json
import time
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import boto3
import pandas as pd
from sklearn.model_selection import train_test_split

# ============================================================
# FINAL TEAM02 V5 CONFIGURATION
# ============================================================

REGION = "ap-southeast-1"
BUCKET = "nyp-26s1-iti113"
TEAM_ID = "team02"
PROJECT_NAME = "diabetes-risk"

WATCH_PREFIX = f"iti113/{TEAM_ID}/trigger/input/"
PIPELINE_NAME = "team02-diabetes-risk"

FINAL_MODEL_PACKAGE_GROUP = "team02-diabetes-risk"
FINAL_ENDPOINT_NAME = "team02-diabetes-risk"

CHAMPION_MODEL_TYPE = "XGBoost"
CHAMPION_FINAL_RUN_ID = "057bb58d81564c0db9a628b224eab8f9"
BEST_MODEL_CONFIG_VERSION = "2026-08-22-cross-model-champion-v5"

EXPERIMENT_PROFILE_ROWS = 253_680
EXPERIMENT_TEST_SIZE = 0.20
EXPERIMENT_RANDOM_STATE = 42

EXPERIMENT_PROFILE_SHA256 = (
    "6244bec277fe3cefce56d46c583a4c5eb73f179ef64c8a1b18557fd0200105c5"
)
EXPERIMENT_TRAIN_SPLIT_SHA256 = (
    "93531a7b366471a63d29fd970abad6c5d04a2f933f9ffb2aa38dce1c7b286d65"
)
EXPERIMENT_TEST_SPLIT_SHA256 = (
    "12de3cf4074e8656a059c938acbe27143794910116dda3dc217a101698a368c1"
)
DATA_VERSION = "brfss2015-diabetes-binary-" + EXPERIMENT_PROFILE_SHA256[:12]

# Final v5 registration quality gate.
MIN_TEST_ROC_AUC = 0.75

FEATURE_COLUMNS = [
    "HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "GenHlth",
    "MentHlth", "PhysHlth", "DiffWalk", "Sex", "Age", "Education", "Income",
]
SOURCE_TARGET_COLUMN = "Diabetes_binary"
EXPECTED_COLUMNS = FEATURE_COLUMNS + [SOURCE_TARGET_COLUMN]

EXPECTED_PIPELINE_PARAMETERS = {
    "InputDataUrl",
    "ExpectedProfileSHA256",
    "ExpectedTrainSplitSHA256",
    "ExpectedTestSplitSHA256",
    "MinTestRocAuc",
}

EXPECTED_PIPELINE_STEPS = {
    "team02-preprocess-data",
    "team02-train-model",
    "team02-evaluate-model",
    "team02-quality-gate",
}

s3 = boto3.client("s3", region_name=REGION)
sm = boto3.client("sagemaker", region_name=REGION)
sts = boto3.client("sts", region_name=REGION)

caller = sts.get_caller_identity()

print("AWS account       :", caller["Account"])
print("Caller ARN        :", caller["Arn"])
print("Region            :", REGION)
print("Bucket            :", BUCKET)
print("Watch prefix      :", WATCH_PREFIX)
print("Pipeline          :", PIPELINE_NAME)
print("Champion          :", CHAMPION_MODEL_TYPE)
print("Champion run ID   :", CHAMPION_FINAL_RUN_ID)
print("Config version    :", BEST_MODEL_CONFIG_VERSION)
print("Data version      :", DATA_VERSION)
print("Package group     :", FINAL_MODEL_PACKAGE_GROUP)
print("Final endpoint    :", FINAL_ENDPOINT_NAME)


AWS account       : 044528205969
Caller ARN        : arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team02/SageMaker
Region            : ap-southeast-1
Bucket            : nyp-26s1-iti113
Watch prefix      : iti113/team02/trigger/input/
Pipeline          : team02-diabetes-risk
Champion          : XGBoost
Champion run ID   : 057bb58d81564c0db9a628b224eab8f9
Config version    : 2026-08-22-cross-model-champion-v5
Data version      : brfss2015-diabetes-binary-6244bec277fe
Package group     : team02-diabetes-risk
Final endpoint    : team02-diabetes-risk


## 3. Verify the AWS Pipeline Before Enabling the Trigger

This is a critical stale-pipeline protection. The notebook checks:

- pipeline name;
- final v5 parameter contract;
- expected pipeline steps;
- Team02 tag;
- XGBoost model tag;
- v5 configuration-version tag.

If an older pipeline is still deployed in SageMaker, the trigger is blocked rather than starting the wrong workflow.


In [3]:
def verify_pipeline_contract():
    desc = sm.describe_pipeline(PipelineName=PIPELINE_NAME)
    definition = json.loads(desc["PipelineDefinition"])

    parameter_names = {
        item["Name"] for item in definition.get("Parameters", [])
    }
    step_names = {
        item["Name"] for item in definition.get("Steps", [])
    }

    pipeline_arn = desc["PipelineArn"]
    tag_items = sm.list_tags(ResourceArn=pipeline_arn).get("Tags", [])
    tags = {item["Key"]: item["Value"] for item in tag_items}

    missing_parameters = EXPECTED_PIPELINE_PARAMETERS - parameter_names
    missing_steps = EXPECTED_PIPELINE_STEPS - step_names

    checks = {
        "pipeline_name": desc.get("PipelineName") == PIPELINE_NAME,
        "required_parameters": not missing_parameters,
        "required_steps": not missing_steps,
        "team_tag": tags.get("TeamId") == TEAM_ID,
        "model_tag": tags.get("ModelType") == CHAMPION_MODEL_TYPE,
        "config_version_tag": (
            tags.get("ConfigVersion") == BEST_MODEL_CONFIG_VERSION
        ),
    }

    print("SAGEMAKER PIPELINE CONTRACT CHECK")
    print("=" * 88)
    print("Pipeline ARN     :", pipeline_arn)
    print("Parameters       :", sorted(parameter_names))
    print("Steps            :", sorted(step_names))
    print("Relevant tags    :", {
        k: tags.get(k)
        for k in ("TeamId", "ModelType", "ConfigVersion", "ProjectName")
    })
    print("Checks           :", checks)

    if missing_parameters:
        print("Missing parameters:", sorted(missing_parameters))
    if missing_steps:
        print("Missing steps     :", sorted(missing_steps))

    if REQUIRE_FINAL_V5_PIPELINE and not all(checks.values()):
        raise RuntimeError(
            "Trigger blocked: the AWS SageMaker Pipeline does not match "
            "the final Team02 v5 contract. Rerun Notebook 03 v5 through "
            "the pipeline upsert section before enabling S3 polling."
        )

    print("PIPELINE CONTRACT: PASS")
    return {
        "pipeline_arn": pipeline_arn,
        "parameters": sorted(parameter_names),
        "steps": sorted(step_names),
        "tags": tags,
        "checks": checks,
    }

pipeline_contract = verify_pipeline_contract()


SAGEMAKER PIPELINE CONTRACT CHECK
Pipeline ARN     : arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/team02-diabetes-risk
Parameters       : ['ExpectedProfileSHA256', 'ExpectedTestSplitSHA256', 'ExpectedTrainSplitSHA256', 'InputDataUrl', 'MinTestRocAuc']
Steps            : ['team02-evaluate-model', 'team02-preprocess-data', 'team02-quality-gate', 'team02-train-model']
Relevant tags    : {'TeamId': 'team02', 'ModelType': 'XGBoost', 'ConfigVersion': '2026-08-22-cross-model-champion-v5', 'ProjectName': 'diabetes-risk'}
Checks           : {'pipeline_name': True, 'required_parameters': True, 'required_steps': True, 'team_tag': True, 'model_tag': True, 'config_version_tag': True}
PIPELINE CONTRACT: PASS


## 4. Dataset-Lineage Preflight

Notebook 03 v5 performs strict lineage checks inside the SageMaker Processing Job. The trigger repeats those checks **before** starting a paid pipeline execution.

A candidate CSV must have:

- exactly 253,680 rows;
- the exact 21-feature + `Diabetes_binary` schema and order;
- binary target values `{0, 1}`;
- the frozen full-profile dataframe SHA-256;
- the frozen deterministic 80/20 train/test SHA-256 signatures.

This means the current trigger demonstrates **controlled pipeline retraining/reproducibility**, not unrestricted learning from arbitrary newly-arriving production data.


In [4]:
def dataframe_sha256(frame: pd.DataFrame) -> str:
    hashed = pd.util.hash_pandas_object(
        frame,
        index=False,
    ).values.tobytes()
    return hashlib.sha256(hashed).hexdigest()


def load_s3_csv(bucket: str, key: str) -> pd.DataFrame:
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    return pd.read_csv(io.BytesIO(body)).reset_index(drop=True)


def validate_candidate_dataset(bucket: str, key: str) -> dict:
    if not key.lower().endswith(".csv"):
        return {
            "valid": False,
            "reason": "Only CSV trigger objects are accepted.",
            "s3_uri": f"s3://{bucket}/{key}",
        }

    df = load_s3_csv(bucket, key)

    if len(df) != EXPERIMENT_PROFILE_ROWS:
        return {
            "valid": False,
            "reason": (
                f"Row count mismatch: expected {EXPERIMENT_PROFILE_ROWS:,}, "
                f"found {len(df):,}."
            ),
            "s3_uri": f"s3://{bucket}/{key}",
        }

    if list(df.columns) != EXPECTED_COLUMNS:
        return {
            "valid": False,
            "reason": (
                "Schema/order mismatch. "
                f"Expected {EXPECTED_COLUMNS}; found {list(df.columns)}."
            ),
            "s3_uri": f"s3://{bucket}/{key}",
        }

    if df.isna().sum().sum() != 0:
        return {
            "valid": False,
            "reason": "Conventional missing values were detected.",
            "s3_uri": f"s3://{bucket}/{key}",
        }

    target_values = sorted(
        pd.to_numeric(df[SOURCE_TARGET_COLUMN], errors="coerce")
        .dropna()
        .unique()
        .tolist()
    )
    if target_values != [0, 1]:
        return {
            "valid": False,
            "reason": f"Unexpected target values: {target_values}.",
            "s3_uri": f"s3://{bucket}/{key}",
        }

    profile_hash = dataframe_sha256(df)
    if profile_hash != EXPERIMENT_PROFILE_SHA256:
        return {
            "valid": False,
            "reason": (
                "Frozen full-profile signature mismatch. "
                f"expected={EXPERIMENT_PROFILE_SHA256}, observed={profile_hash}"
            ),
            "s3_uri": f"s3://{bucket}/{key}",
            "profile_sha256": profile_hash,
        }

    X = df[FEATURE_COLUMNS].copy()
    y = df[SOURCE_TARGET_COLUMN].astype(int).copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=EXPERIMENT_TEST_SIZE,
        random_state=EXPERIMENT_RANDOM_STATE,
        stratify=y,
    )

    train_for_hash = pd.concat(
        [
            X_train.reset_index(drop=True),
            y_train.reset_index(drop=True).rename(SOURCE_TARGET_COLUMN),
        ],
        axis=1,
    )
    test_for_hash = pd.concat(
        [
            X_test.reset_index(drop=True),
            y_test.reset_index(drop=True).rename(SOURCE_TARGET_COLUMN),
        ],
        axis=1,
    )

    train_hash = dataframe_sha256(train_for_hash)
    test_hash = dataframe_sha256(test_for_hash)

    if train_hash != EXPERIMENT_TRAIN_SPLIT_SHA256:
        return {
            "valid": False,
            "reason": (
                "Frozen train-split signature mismatch. "
                f"expected={EXPERIMENT_TRAIN_SPLIT_SHA256}, observed={train_hash}"
            ),
            "s3_uri": f"s3://{bucket}/{key}",
        }

    if test_hash != EXPERIMENT_TEST_SPLIT_SHA256:
        return {
            "valid": False,
            "reason": (
                "Frozen test-split signature mismatch. "
                f"expected={EXPERIMENT_TEST_SPLIT_SHA256}, observed={test_hash}"
            ),
            "s3_uri": f"s3://{bucket}/{key}",
        }

    return {
        "valid": True,
        "reason": "Exact final-v5 dataset and split lineage reproduced.",
        "s3_uri": f"s3://{bucket}/{key}",
        "rows": len(df),
        "profile_sha256": profile_hash,
        "train_split_sha256": train_hash,
        "test_split_sha256": test_hash,
        "data_version": DATA_VERSION,
    }

print("Dataset-lineage validator ready.")


Dataset-lineage validator ready.


## 5. S3 Object Detection and Audit Trail

The original polling notebook tracked only object keys. This version tracks an object fingerprint using **key + ETag + size + last-modified time**, so replacing a file at the same S3 key can still be detected.

Every trigger decision is appended to a local JSONL audit file for reproducibility evidence.


In [5]:
AUDIT_PATH = Path("team02_s3_polling_trigger_audit.jsonl")


def list_trigger_objects():
    paginator = s3.get_paginator("list_objects_v2")
    objects = []

    for page in paginator.paginate(
        Bucket=BUCKET,
        Prefix=WATCH_PREFIX,
    ):
        for obj in page.get("Contents", []):
            key = obj["Key"]

            if key.endswith("/"):
                continue

            objects.append({
                "key": key,
                "etag": obj.get("ETag", "").strip('"'),
                "size": int(obj.get("Size", 0)),
                "last_modified": obj["LastModified"].astimezone(
                    timezone.utc
                ).isoformat(),
            })

    return objects


def object_fingerprint(obj):
    raw = (
        f'{obj["key"]}|{obj["etag"]}|'
        f'{obj["size"]}|{obj["last_modified"]}'
    )
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def append_audit(record):
    record = dict(record)
    record.setdefault(
        "audit_timestamp_utc",
        datetime.now(timezone.utc).isoformat(),
    )
    with AUDIT_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, default=str) + "\n")


def read_audit():
    if not AUDIT_PATH.exists():
        return pd.DataFrame()

    rows = []
    with AUDIT_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


objects_now = list_trigger_objects()

if PROCESS_EXISTING_OBJECTS:
    seen_fingerprints = set()
    print(
        "Existing objects WILL be treated as candidates on the first scan:",
        len(objects_now),
    )
else:
    seen_fingerprints = {
        object_fingerprint(obj)
        for obj in objects_now
    }
    print(
        "Initialized baseline. Existing objects ignored:",
        len(seen_fingerprints),
    )

print("Watching:", f"s3://{BUCKET}/{WATCH_PREFIX}")
print("Audit file:", AUDIT_PATH)


Initialized baseline. Existing objects ignored: 2
Watching: s3://nyp-26s1-iti113/iti113/team02/trigger/input/
Audit file: team02_s3_polling_trigger_audit.jsonl


## 6. Start the Final v5 Pipeline

For every valid new object the trigger supplies all final-v5 pipeline parameters:

- `InputDataUrl`
- `ExpectedProfileSHA256`
- `ExpectedTrainSplitSHA256`
- `ExpectedTestSplitSHA256`
- `MinTestRocAuc`

The resulting execution ARN is captured in the audit trail.

**Release-control boundary:** a successful pipeline run may create a new model package, but the v5 pipeline registers it as `PendingManualApproval`. This notebook intentionally contains no auto-approval and no endpoint-deployment command.


In [6]:
def start_final_v5_pipeline(input_data_url: str) -> dict:
    execution_parameters = {
        "InputDataUrl": input_data_url,
        "ExpectedProfileSHA256": EXPERIMENT_PROFILE_SHA256,
        "ExpectedTrainSplitSHA256": EXPERIMENT_TRAIN_SPLIT_SHA256,
        "ExpectedTestSplitSHA256": EXPERIMENT_TEST_SPLIT_SHA256,
        "MinTestRocAuc": str(MIN_TEST_ROC_AUC),
    }

    display_name = (
        "team02-s3-trigger-"
        + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    )

    print("Starting SageMaker Pipeline")
    print("=" * 88)
    print("Pipeline   :", PIPELINE_NAME)
    print("Input      :", input_data_url)
    print("Display    :", display_name)
    print("Parameters :")
    print(json.dumps(execution_parameters, indent=2))

    response = sm.start_pipeline_execution(
        PipelineName=PIPELINE_NAME,
        PipelineExecutionDisplayName=display_name,
        PipelineExecutionDescription=(
            "Team02 controlled S3 polling trigger for final v5 champion pipeline"
        ),
        PipelineParameters=[
            {"Name": name, "Value": str(value)}
            for name, value in execution_parameters.items()
        ],
    )

    execution_arn = response["PipelineExecutionArn"]
    print("Pipeline execution started:", execution_arn)

    return {
        "pipeline_execution_arn": execution_arn,
        "execution_parameters": execution_parameters,
        "display_name": display_name,
    }


def get_execution_status(execution_arn: str) -> dict:
    desc = sm.describe_pipeline_execution(
        PipelineExecutionArn=execution_arn
    )
    return {
        "pipeline_execution_arn": execution_arn,
        "status": desc.get("PipelineExecutionStatus"),
        "start_time": desc.get("CreationTime"),
        "last_modified_time": desc.get("LastModifiedTime"),
        "failure_reason": desc.get("FailureReason"),
    }

print("Pipeline trigger function ready.")


Pipeline trigger function ready.


## 7. Poll Once

Use this cell first. It makes one scan only, which is easier to test and demonstrate than immediately entering an infinite loop.

A newly detected object follows this decision path:

`new object → frozen-lineage preflight → PASS: start pipeline / FAIL: block and log`


In [7]:
def poll_once():
    global seen_fingerprints

    objects = list_trigger_objects()
    detected = 0

    for obj in objects:
        fingerprint = object_fingerprint(obj)

        if fingerprint in seen_fingerprints:
            continue

        detected += 1
        seen_fingerprints.add(fingerprint)

        s3_uri = f's3://{BUCKET}/{obj["key"]}'

        print("\n" + "=" * 88)
        print("NEW S3 OBJECT DETECTED")
        print("S3 URI       :", s3_uri)
        print("ETag         :", obj["etag"])
        print("Size         :", obj["size"])
        print("Last modified:", obj["last_modified"])

        audit_record = {
            "event": "s3_object_detected",
            "pipeline_name": PIPELINE_NAME,
            "config_version": BEST_MODEL_CONFIG_VERSION,
            "champion_model": CHAMPION_MODEL_TYPE,
            "champion_run_id": CHAMPION_FINAL_RUN_ID,
            "data_version": DATA_VERSION,
            "s3_uri": s3_uri,
            "object_fingerprint": fingerprint,
        }

        try:
            preflight = validate_candidate_dataset(
                BUCKET,
                obj["key"],
            )

            print("Preflight valid :", preflight["valid"])
            print("Preflight reason:", preflight["reason"])

            audit_record["preflight"] = preflight

            if not preflight["valid"]:
                audit_record["action"] = "BLOCKED"
                audit_record["reason"] = preflight["reason"]
                append_audit(audit_record)
                continue

            trigger_result = start_final_v5_pipeline(s3_uri)

            audit_record["action"] = "PIPELINE_STARTED"
            audit_record.update(trigger_result)
            append_audit(audit_record)

        except Exception as exc:
            audit_record["action"] = "ERROR"
            audit_record["reason"] = repr(exc)
            append_audit(audit_record)

            print("Trigger processing failed:")
            print(repr(exc))

    if detected == 0:
        print(
            datetime.now(timezone.utc).isoformat(),
            "- no new S3 objects detected.",
        )
    else:
        print("\nObjects processed this scan:", detected)

    return detected


new_objects = poll_once()


2026-08-22T13:09:13.673926+00:00 - no new S3 objects detected.


## 8. Optional Continuous Polling

Set `RUN_CONTINUOUS_POLLING = True` in the Human Control Panel before running this cell.

Stop the cell using **Interrupt / Stop Kernel Execution**. The trigger cannot run while the notebook kernel is stopped; this is why it is suitable as a course-demo alternative to Lambda/EventBridge, not as a production event architecture.


In [8]:
# Keep False while validating the notebook.
# Set True only when you intentionally want the cell to remain polling.
RUN_CONTINUOUS_POLLING = True

In [9]:
if not RUN_CONTINUOUS_POLLING:
    print("Continuous polling is disabled.")
    print(
        "Set RUN_CONTINUOUS_POLLING=True in the Human Control Panel "
        "and rerun this cell when ready."
    )
else:
    print("CONTINUOUS S3 POLLING STARTED")
    print("=" * 88)
    print("Watching :", f"s3://{BUCKET}/{WATCH_PREFIX}")
    print("Pipeline :", PIPELINE_NAME)
    print("Interval :", CHECK_INTERVAL_SECONDS, "seconds")
    print("Stop     : interrupt this notebook cell")
    print("=" * 88)

    try:
        while True:
            poll_once()
            print(
                f"Polling again in {CHECK_INTERVAL_SECONDS} seconds..."
            )
            time.sleep(CHECK_INTERVAL_SECONDS)

    except KeyboardInterrupt:
        print("\nPolling stopped by user.")


CONTINUOUS S3 POLLING STARTED
Watching : s3://nyp-26s1-iti113/iti113/team02/trigger/input/
Pipeline : team02-diabetes-risk
Interval : 60 seconds
Stop     : interrupt this notebook cell
2026-08-22T13:09:40.353207+00:00 - no new S3 objects detected.
Polling again in 60 seconds...
2026-08-22T13:10:40.435150+00:00 - no new S3 objects detected.
Polling again in 60 seconds...
2026-08-22T13:11:40.476770+00:00 - no new S3 objects detected.
Polling again in 60 seconds...
2026-08-22T13:12:40.522834+00:00 - no new S3 objects detected.
Polling again in 60 seconds...

NEW S3 OBJECT DETECTED
S3 URI       : s3://nyp-26s1-iti113/iti113/team02/trigger/input/diabetes_binary_cleaned_dataset_trigger_20260822_131302.csv
ETag         : b03e0bbb56fbcb905633f90ea859e3ad
Size         : 22230794
Last modified: 2026-08-22T13:13:03+00:00
Preflight valid : True
Preflight reason: Exact final-v5 dataset and split lineage reproduced.
Starting SageMaker Pipeline
Pipeline   : team02-diabetes-risk
Input      : s3://nyp-

## 9. Review Trigger Audit and Pipeline Execution

This evidence helps demonstrate **traceability and reproducibility** for the MLOps report:

- which S3 object caused the event;
- the v5 configuration/model/data identity;
- whether lineage preflight passed;
- the SageMaker execution ARN;
- whether the trigger was blocked.

Use the execution ARN with SageMaker Pipeline execution history for training/evaluation/registration evidence.


In [10]:
audit_df = read_audit()

if audit_df.empty:
    print("No trigger audit records have been written yet.")
else:
    display(audit_df)

    started = audit_df[
        audit_df["action"] == "PIPELINE_STARTED"
    ] if "action" in audit_df.columns else pd.DataFrame()

    if not started.empty:
        latest_arn = started.iloc[-1]["pipeline_execution_arn"]
        print("\nLatest triggered execution:")
        print(json.dumps(
            get_execution_status(latest_arn),
            indent=2,
            default=str,
        ))


,event,pipeline_name,config_version,champion_model,champion_run_id,data_version,s3_uri,object_fingerprint,preflight,action,pipeline_execution_arn,execution_parameters,display_name,audit_timestamp_utc
0,s3_object_detected,team02-diabetes-risk,2026-08-22-cross-model-champion-v5,XGBoost,057bb58d81564c0db9a628b224eab8f9,brfss2015-diabetes-binary-6244bec277fe,s3://nyp-26s1-iti113/iti113/team02/trigger/inp...,7e815be3eb7b54b5e9fd1cd07e1a9aef54aada884b6927...,"{'valid': True, 'reason': 'Exact final-v5 data...",PIPELINE_STARTED,arn:aws:sagemaker:ap-southeast-1:044528205969:...,{'InputDataUrl': 's3://nyp-26s1-iti113/iti113/...,team02-s3-trigger-20260822-131341,2026-08-22T13:13:42.081523+00:00



Latest triggered execution:
{
  "pipeline_execution_arn": "arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/team02-diabetes-risk/execution/smp86sip3ek6",
  "status": "Succeeded",
  "start_time": "2026-08-22 13:13:41.956000+00:00",
  "last_modified_time": "2026-08-22 13:26:52.259000+00:00",
  "failure_reason": null
}


## 10. Demo Procedure

1. Run Notebook 03 v5 through the **pipeline upsert** section so AWS contains the final `team02-diabetes-risk` definition.
2. Run Sections 1–6 of this trigger notebook.
3. Confirm **`PIPELINE CONTRACT: PASS`**.
4. Leave `PROCESS_EXISTING_OBJECTS=False`.
5. Run **Poll Once** once to establish that there are no new files.
6. Set `RUN_CONTINUOUS_POLLING=True` and run the continuous polling cell.
7. Upload a **copy of the approved cleaned dataset** to:
   `s3://nyp-26s1-iti113/iti113/team02/trigger/input/`
8. The notebook detects the upload and performs the frozen lineage preflight.
9. On PASS, it starts `team02-diabetes-risk` with all five v5 parameters.
10. Open SageMaker → Pipelines → `team02-diabetes-risk` and show the new execution.
11. After success, show the newly registered model package as **PendingManualApproval**.
12. Perform the separate human review/approval and deployment procedure from Notebook 03 v5 when appropriate.

## Why arbitrary changed data is blocked

The v5 pipeline was built to reproduce the **final assessed champion** against an exact frozen experiment dataset and split. Therefore an arbitrary new production dataset will fail the profile/split signatures.

For a future production continuous-training design, the pipeline should use a different governed change-management workflow: version the incoming dataset, run schema/quality/drift checks, create a new train/validation/test lineage, evaluate champion-vs-challenger performance and fairness, then require human approval before replacement. Do **not** simply disable the v5 hash controls.

## Assessment mapping

**C — MLOps & Deployment**
- S3 data/change detection;
- automated SageMaker Pipeline invocation;
- complete parameter handoff;
- reproducible trigger audit;
- model registry handoff;
- human-controlled release boundary.

**E — AI Governance**
- input lineage is validated before training;
- stale pipeline versions are blocked;
- arbitrary changed data cannot silently replace the assessed dataset;
- model registration remains `PendingManualApproval`;
- no automated approval/deployment;
- auditable trigger → pipeline execution linkage.

This aligns the trigger notebook with the final MLOps architecture while preserving the responsible-AI controls in Notebook 03 v5.
